# Notebook 1 — A real bank file, not a toy schema


## The file

| What | Where |
|---|---|
| Database | `data/financial.sqlite`, 8 tables, about 1 million transactions |
| Source | [domyn/FINCH](https://huggingface.co/datasets/domyn/FINCH) on Hugging Face, the BIRD database named `financial` |
| Origin | Petr Berka's Czech bank, published for PKDD 1999 |
| Licence | CC-BY-NC 4.0. For study. Do not ship the file inside a paid product |
| Dictionary | `semantic_layer.json`. This is the semantic layer a data team keeps beside the warehouse |

Meera works a branch desk. A client asks how many accounts are in Pisek. The district name is in a column called `A2`. The statement frequency is the Czech code `POPLATEK PO OBRATU`. A model that guesses English words gets zero rows, and a count of zero looks like an answer.

## Outline

1. Open the file and count the rows.
2. Read the dictionary next to the raw column names.
3. Run the lowercase filter and the stored spelling.
4. See why `%y` is the wrong year code.

Open `semantic_layer.json` and the value catalog. The database stores the code. The dictionary says what the code means.


## Step 0 — Setup

Ollama is not required in this notebook. The SQL runs locally.

If `data/financial.sqlite` is missing, run `python download_corpus.py` in this folder. That copies one 71 MB file out of FINCH. The other databases in that dataset stay in the Hugging Face cache.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "text2sql.py").exists():
    raise SystemExit("Set the notebook's working directory to 02_bank_book.")
sys.path.insert(0, str(HERE))

import text2sql as desk

connection = desk.connect()
tables = [row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1")]
print("tables:", ", ".join(tables))
for name in tables:
    count = connection.execute(f"SELECT COUNT(*) FROM {desk.qtable(name)}").fetchone()[0]
    print(f"  {name:10} {count:,} rows")
connection.close()


tables: account, card, client, disp, district, loan, order, trans
  account    4,500 rows
  card       892 rows
  client     5,369 rows
  disp       5,369 rows
  district   77 rows
  loan       682 rows
  order      6,471 rows
  trans      1,056,320 rows


## Step 1 — The column name is not the meaning

`district` has columns `A2`, `A3`, `A11`, `A12`. Nothing in the SQLite catalog says that `A11` is the average salary. Production text-to-SQL does not hand the model a bare `CREATE TABLE` and hope. It hands over a semantic layer: one sentence per column, plus the codes that are actually stored.

`trans` has 1,056,320 rows. A question that does not need a transaction must not drag that table into the prompt.


In [2]:
layer = desk.load_layer()
for column, meaning in layer["tables"]["district"]["columns"].items():
    print(f"{column:12} {meaning}")

print()
catalog = desk.value_catalog()
print("account.frequency")
for line in catalog["account.frequency"]:
    print(" ", line)
print("district.A3")
for line in catalog["district.A3"]:
    print(" ", line)


district_id  Primary key. Branch and client tables point here.
A2           District name, for example Pisek or Hl.m. Praha.
A3           Region name, for example east Bohemia or Prague.
A4           Number of inhabitants. Stored as text in this file.
A5           Municipalities with fewer than 499 inhabitants.
A6           Municipalities with 500 to 1999 inhabitants.
A7           Municipalities with 2000 to 9999 inhabitants.
A8           Municipalities with 10000 or more inhabitants.
A9           Number of cities.
A10          Ratio of urban inhabitants.
A11          Average salary.
A12          Unemployment rate in 1995.
A13          Unemployment rate in 1996.
A14          Entrepreneurs per 1000 inhabitants.
A15          Number of crimes committed in 1995.
A16          Number of crimes committed in 1996.

account.frequency
  POPLATEK MESICNE = monthly statement
  POPLATEK TYDNE = weekly statement
  POPLATEK PO OBRATU = statement issued after each transaction
district.A3
  south Morav

## Step 2 — The published spelling returns zero

FINCH writes `east bohemia` and `poplatek po obratu`. This file stores `east Bohemia` and `POPLATEK PO OBRATU`. SQLite `=` is case-sensitive. A count of zero is a row, so a careless check says the query succeeded.

When a count comes back as 0, check the stored values before you send that 0. Zero often means the filter string is not in the file.


In [3]:
published = """
SELECT COUNT(*) AS accounts
FROM account AS a
INNER JOIN district AS d ON a.district_id = d.district_id
WHERE d.A3 = 'east bohemia' AND a.frequency = 'poplatek po obratu'
"""
stored = """
SELECT COUNT(*) AS accounts
FROM account AS a
INNER JOIN district AS d ON a.district_id = d.district_id
WHERE d.A3 = 'east Bohemia' AND a.frequency = 'POPLATEK PO OBRATU'
"""
print("published spelling:", desk.execute_sql(published)["rows"])
print("stored spelling:   ", desk.execute_sql(stored)["rows"])


published spelling: [[0]]
stored spelling:    [[13]]


## Step 3 — The year code

Dates in this file look like `1993-01-01`.

`strftime('%Y', date)` returns `1993`.
`strftime('%y', date)` returns `93`.

A filter that compares `%y` with `1993` returns zero accounts. The accounts are there.


In [4]:
print("opened in 1993 with %Y:", desk.execute_sql("SELECT COUNT(*) FROM account WHERE strftime('%Y', date) = '1993'")["rows"])
print("same filter with %y:   ", desk.execute_sql("SELECT COUNT(*) FROM account WHERE strftime('%y', date) = '1993'")["rows"])


opened in 1993 with %Y: [[1139]]
same filter with %y:    [[0]]
